# 🤖 Multi-Agent Trading Bot — Interactive Colab Notebook

**4-Agent Hybrid Trading System for Indian Equities & Crypto**

```
Agent 1 (Technical Observer) ──┐
                                ├──► Agent 3 (Strategy Engine) ──► Agent 4 (Risk & Execution)
Agent 2 (Sentiment Researcher) ┘
```

| Agent | Role | Technology |
|-------|------|------------|
| Agent 1 | RSI, SMA, MACD, ATR, Candlestick Patterns | Pure Python (pandas, ta) |
| Agent 2 | News Sentiment Scoring (0–100) | Google Gemini AI + RSS |
| Agent 3 | Strategy Selection (Breakout / Reversion / Hold) | Deterministic Playbook |
| Agent 4 | 2% Risk Rule, Position Sizing, Order Dispatch | Hard-Coded Guardrails |

---
**Run each cell in order ⬇️**

## 📦 Step 1: Install Dependencies

In [ ]:
!pip install -q yfinance ccxt pandas numpy ta google-genai feedparser httpx

## 🔑 Step 2: Configure API Keys & Settings

Set your **Gemini API key** below. Get a free one from [Google AI Studio](https://aistudio.google.com/apikey).

In [ ]:
# ══════════════════════════════════════════════════════════
#  CONFIGURATION — Edit these values
# ══════════════════════════════════════════════════════════

GEMINI_API_KEY = ""  # 🔑 Paste your Gemini API key here

# Execution mode
EXECUTION_MODE = "PAPER"  # "PAPER" (simulation) or "LIVE"

# Capital
DEFAULT_CAPITAL_INR = 100_000  # ₹1,00,000
DEFAULT_CAPITAL_USD = 10_000   # $10,000

# Risk parameters
MAX_RISK_PER_TRADE = 0.02   # 2% max risk per trade
MIN_RISK_REWARD = 2.0       # Minimum 1:2 R:R ratio
MAX_DAILY_DRAWDOWN = 0.06   # 6% daily drawdown halt

# Technical indicator periods
RSI_PERIOD = 14
SMA_SHORT = 20
SMA_MEDIUM = 50
SMA_LONG = 200
ATR_PERIOD = 14
MACD_FAST = 12
MACD_SLOW = 26
MACD_SIGNAL = 9
BOLLINGER_PERIOD = 20
BOLLINGER_STD = 2

# Sentiment thresholds
SENTIMENT_BULLISH_THRESHOLD = 60
SENTIMENT_BEARISH_THRESHOLD = 40

# Strategy thresholds
MOMENTUM_RSI_CEILING = 70
REVERSION_RSI_FLOOR = 30
DEFAULT_SL_PERCENT = 0.05
DEFAULT_TP_PERCENT = 0.10

# Watchlists
INDIAN_WATCHLIST = [
    "RELIANCE.NS", "TCS.NS", "HDFCBANK.NS", "INFY.NS", "TATASTEEL.NS",
    "ICICIBANK.NS", "SBIN.NS", "BHARTIARTL.NS", "ITC.NS", "WIPRO.NS",
]

CRYPTO_WATCHLIST = [
    "BTC/USDT", "ETH/USDT", "SOL/USDT", "XRP/USDT", "BNB/USDT",
]

# RSS News Feeds
NEWS_RSS_FEEDS = {
    "moneycontrol": "https://www.moneycontrol.com/rss/latestnews.xml",
    "economic_times_markets": "https://economictimes.indiatimes.com/markets/rssfeeds/1977021501.cms",
    "livemint_markets": "https://www.livemint.com/rss/markets",
    "cryptopanic": "https://cryptopanic.com/news/rss/",
}

print("✅ Configuration loaded successfully!")
print(f"   Mode: {EXECUTION_MODE}")
print(f"   Indian Capital: ₹{DEFAULT_CAPITAL_INR:,.0f}")
print(f"   Crypto Capital: ${DEFAULT_CAPITAL_USD:,.0f}")
print(f"   Max Risk/Trade: {MAX_RISK_PER_TRADE*100}%")

---
## 📊 Step 3: Market Data Connectors

In [ ]:
# ══════════════════════════════════════════════════════════
#  MARKET DATA CONNECTORS
# ══════════════════════════════════════════════════════════

import yfinance as yf
import ccxt
import pandas as pd
from datetime import datetime, timedelta, timezone


def fetch_indian_ohlcv(symbol: str, period_days: int = 60) -> pd.DataFrame:
    """Fetch OHLCV data for an Indian equity ticker via Yahoo Finance."""
    end_date = datetime.now()
    start_date = end_date - timedelta(days=period_days)
    ticker = yf.Ticker(symbol)
    df = ticker.history(start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'), interval='1d')
    if df.empty:
        raise ValueError(f"No data for {symbol}")
    df.columns = [col.lower() for col in df.columns]
    return df[['open', 'high', 'low', 'close', 'volume']].dropna()


def fetch_crypto_ohlcv(symbol: str, timeframe: str = '1d', limit: int = 60) -> pd.DataFrame:
    """Fetch OHLCV data for a crypto pair via CCXT (Binance)."""
    exchange = ccxt.binance({'enableRateLimit': True})
    raw = exchange.fetch_ohlcv(symbol, timeframe=timeframe, limit=limit)
    if not raw:
        raise ValueError(f"No data for {symbol}")
    df = pd.DataFrame(raw, columns=['timestamp', 'open', 'high', 'low', 'close', 'volume'])
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='ms', utc=True)
    df.set_index('timestamp', inplace=True)
    return df


def get_indian_price(symbol: str) -> float:
    """Get latest price for Indian equity."""
    return float(yf.Ticker(symbol).history(period='1d')['Close'].iloc[-1])


def get_crypto_price(symbol: str) -> float:
    """Get latest price for crypto pair."""
    return float(ccxt.binance({'enableRateLimit': True}).fetch_ticker(symbol)['last'])


# Quick test
print("✅ Market data connectors ready!")
print(f"   Testing RELIANCE.NS...")
test_df = fetch_indian_ohlcv("RELIANCE.NS", period_days=10)
print(f"   ✅ Got {len(test_df)} candles for RELIANCE.NS")
print(test_df.tail(3))

---
## 🔬 Step 4: Agent 1 — Technical & Market Observer
Pure deterministic math — zero LLM involvement.

In [ ]:
# ══════════════════════════════════════════════════════════
#  AGENT 1: TECHNICAL & MARKET OBSERVER
# ══════════════════════════════════════════════════════════

import numpy as np
import ta


def compute_indicators(df: pd.DataFrame) -> pd.DataFrame:
    """Compute RSI, SMA, MACD, ATR, Bollinger Bands on OHLCV data."""
    close, high, low = df['close'], df['high'], df['low']

    df['rsi'] = ta.momentum.RSIIndicator(close=close, window=RSI_PERIOD).rsi()
    df['sma_20'] = ta.trend.SMAIndicator(close=close, window=SMA_SHORT).sma_indicator()
    df['sma_50'] = ta.trend.SMAIndicator(close=close, window=SMA_MEDIUM).sma_indicator()
    df['sma_200'] = ta.trend.SMAIndicator(close=close, window=SMA_LONG).sma_indicator()

    macd_obj = ta.trend.MACD(close=close, window_slow=MACD_SLOW, window_fast=MACD_FAST, window_sign=MACD_SIGNAL)
    df['macd'] = macd_obj.macd()
    df['macd_signal'] = macd_obj.macd_signal()
    df['macd_histogram'] = macd_obj.macd_diff()

    df['atr'] = ta.volatility.AverageTrueRange(high=high, low=low, close=close, window=ATR_PERIOD).average_true_range()

    bb = ta.volatility.BollingerBands(close=close, window=BOLLINGER_PERIOD, window_dev=BOLLINGER_STD)
    df['bb_upper'] = bb.bollinger_hband()
    df['bb_middle'] = bb.bollinger_mavg()
    df['bb_lower'] = bb.bollinger_lband()

    return df


def detect_candlestick_patterns(df: pd.DataFrame) -> list:
    """Detect common candlestick patterns from last 3 candles."""
    patterns = []
    if len(df) < 3:
        return patterns

    curr, prev = df.iloc[-1], df.iloc[-2]
    body_curr = curr['close'] - curr['open']
    body_prev = prev['close'] - prev['open']
    upper_wick = curr['high'] - max(curr['open'], curr['close'])
    lower_wick = min(curr['open'], curr['close']) - curr['low']
    body_size = abs(body_curr)
    candle_range = curr['high'] - curr['low']

    if candle_range > 0 and body_size / candle_range < 0.1:
        patterns.append('Doji')
    if body_size > 0 and lower_wick > 2 * body_size and upper_wick < body_size:
        patterns.append('Hammer')
    if body_size > 0 and upper_wick > 2 * body_size and lower_wick < body_size:
        patterns.append('Shooting Star')
    if body_prev < 0 and body_curr > 0 and curr['open'] <= prev['close'] and curr['close'] >= prev['open']:
        patterns.append('Bullish Engulfing')
    if body_prev > 0 and body_curr < 0 and curr['open'] >= prev['close'] and curr['close'] <= prev['open']:
        patterns.append('Bearish Engulfing')

    return patterns


def determine_trend(df: pd.DataFrame) -> str:
    """Determine macro trend from SMA alignment."""
    curr = df.iloc[-1]
    sma_20, sma_50, price = curr.get('sma_20'), curr.get('sma_50'), curr['close']
    if pd.isna(sma_20) or pd.isna(sma_50):
        return 'SIDEWAYS'
    if price > sma_20 > sma_50:
        return 'BULLISH'
    elif price < sma_20 < sma_50:
        return 'BEARISH'
    return 'SIDEWAYS'


def detect_breakout(df: pd.DataFrame) -> dict:
    """Detect SMA breakout/breakdown."""
    if len(df) < 2:
        return {'breakout': False, 'direction': 'NONE'}
    curr, prev = df.iloc[-1], df.iloc[-2]
    if pd.notna(curr.get('sma_20')) and pd.notna(prev.get('sma_20')):
        if prev['close'] < prev['sma_20'] and curr['close'] > curr['sma_20']:
            return {'breakout': True, 'direction': 'BULLISH_BREAKOUT_SMA20'}
        if prev['close'] > prev['sma_20'] and curr['close'] < curr['sma_20']:
            return {'breakout': True, 'direction': 'BEARISH_BREAKDOWN_SMA20'}
    return {'breakout': False, 'direction': 'NONE'}


def compute_technical_score(df: pd.DataFrame) -> float:
    """Calculate normalized Technical Score from -1.0 to +1.0."""
    score = 0.0
    curr = df.iloc[-1]
    rsi = curr.get('rsi')
    if pd.notna(rsi):
        if rsi < 30: score += 0.2
        elif rsi < 45: score += 0.1
        elif rsi > 70: score -= 0.3
        elif rsi > 55: score += 0.15

    macd_hist = curr.get('macd_histogram')
    if pd.notna(macd_hist):
        score += 0.25 if macd_hist > 0 else -0.25

    trend = determine_trend(df)
    if trend == 'BULLISH': score += 0.25
    elif trend == 'BEARISH': score -= 0.25

    breakout = detect_breakout(df)
    if breakout['breakout']:
        score += 0.2 if 'BULLISH' in breakout['direction'] else -0.2

    return max(-1.0, min(1.0, round(score, 4)))


def agent1_analyze(df: pd.DataFrame, symbol: str = '') -> dict:
    """Full Agent 1 pipeline."""
    df = compute_indicators(df)
    latest = df.iloc[-1]
    return {
        'agent': 'Technical Observer (Agent 1)',
        'symbol': symbol,
        'latest_price': round(float(latest['close']), 4),
        'indicators': {
            'rsi_14': round(float(latest['rsi']), 2) if pd.notna(latest.get('rsi')) else None,
            'sma_20': round(float(latest['sma_20']), 4) if pd.notna(latest.get('sma_20')) else None,
            'sma_50': round(float(latest['sma_50']), 4) if pd.notna(latest.get('sma_50')) else None,
            'sma_200': round(float(latest['sma_200']), 4) if pd.notna(latest.get('sma_200')) else None,
            'macd': round(float(latest['macd']), 4) if pd.notna(latest.get('macd')) else None,
            'macd_signal': round(float(latest['macd_signal']), 4) if pd.notna(latest.get('macd_signal')) else None,
            'macd_histogram': round(float(latest['macd_histogram']), 4) if pd.notna(latest.get('macd_histogram')) else None,
            'atr_14': round(float(latest['atr']), 4) if pd.notna(latest.get('atr')) else None,
            'bollinger_upper': round(float(latest['bb_upper']), 4) if pd.notna(latest.get('bb_upper')) else None,
            'bollinger_lower': round(float(latest['bb_lower']), 4) if pd.notna(latest.get('bb_lower')) else None,
        },
        'trend': determine_trend(df),
        'breakout': detect_breakout(df),
        'candlestick_patterns': detect_candlestick_patterns(df),
        'technical_score': compute_technical_score(df),
    }


print('✅ Agent 1 (Technical Observer) ready!')

---
## 📰 Step 5: Agent 2 — Macro & Sentiment Researcher
LLM-powered news analysis using Google Gemini AI.

In [ ]:
# ══════════════════════════════════════════════════════════
#  AGENT 2: MACRO & SENTIMENT RESEARCHER
# ══════════════════════════════════════════════════════════

import httpx
import feedparser
from google import genai


def fetch_news_headlines(market_type: str = 'indian', max_headlines: int = 15) -> list:
    """Fetch financial news from RSS feeds."""
    feeds_to_use = {}
    if market_type in ('indian', 'all'):
        feeds_to_use['moneycontrol'] = NEWS_RSS_FEEDS.get('moneycontrol', '')
        feeds_to_use['economic_times_markets'] = NEWS_RSS_FEEDS.get('economic_times_markets', '')
        feeds_to_use['livemint_markets'] = NEWS_RSS_FEEDS.get('livemint_markets', '')
    if market_type in ('crypto', 'all'):
        feeds_to_use['cryptopanic'] = NEWS_RSS_FEEDS.get('cryptopanic', '')

    headlines = []
    for source, url in feeds_to_use.items():
        if not url:
            continue
        try:
            resp = httpx.get(url, timeout=10.0, follow_redirects=True)
            feed = feedparser.parse(resp.text)
            for entry in feed.entries[:max_headlines]:
                title = entry.get('title', '').strip()
                if title:
                    headlines.append(f'[{source}] {title}')
        except Exception:
            continue
    return headlines[:max_headlines]


def analyze_sentiment_with_gemini(headlines: list, symbol: str = '') -> dict:
    """Use Gemini AI to score sentiment 0–100."""
    if not headlines:
        return {'sentiment_score': 50, 'sentiment_label': 'NEUTRAL',
                'narrative_summary': 'No news headlines available.', 'headlines_analyzed': 0}

    headlines_text = '\n'.join(f'- {h}' for h in headlines)
    prompt = f"""You are a professional financial market sentiment analyst.

Analyze the following financial news headlines and determine the overall market sentiment.
{'Asset in focus: ' + symbol if symbol else ''}

NEWS HEADLINES:
{headlines_text}

INSTRUCTIONS:
1. Score from 0 to 100 (0=Extreme Panic, 100=Extreme Greed).
2. Label: EXTREME_PANIC / BEARISH / NEUTRAL / BULLISH / EXTREME_GREED
3. 2-sentence summary explaining WHY.

RESPOND EXACTLY:
SCORE: <number>
LABEL: <label>
SUMMARY: <text>"""

    try:
        client = genai.Client(api_key=GEMINI_API_KEY)
        response = client.models.generate_content(model='gemini-2.5-flash', contents=prompt)
        return _parse_gemini_response(response.text, len(headlines))
    except Exception as e:
        return {'sentiment_score': 50, 'sentiment_label': 'NEUTRAL',
                'narrative_summary': f'Gemini API error: {str(e)[:80]}', 'headlines_analyzed': len(headlines)}


def _parse_gemini_response(text: str, count: int) -> dict:
    """Parse structured Gemini response."""
    score, label, summary = 50, 'NEUTRAL', 'Unable to parse.'
    for line in text.strip().split('\n'):
        line = line.strip()
        if line.upper().startswith('SCORE:'):
            try: score = max(0, min(100, int(float(line.split(':', 1)[1].strip()))))
            except: pass
        elif line.upper().startswith('LABEL:'): label = line.split(':', 1)[1].strip().upper()
        elif line.upper().startswith('SUMMARY:'): summary = line.split(':', 1)[1].strip()
    return {'sentiment_score': score, 'sentiment_label': label,
            'narrative_summary': summary, 'headlines_analyzed': count}


def agent2_analyze(symbol: str = '', market_type: str = 'indian') -> dict:
    """Full Agent 2 pipeline."""
    headlines = fetch_news_headlines(market_type=market_type)
    sentiment = analyze_sentiment_with_gemini(headlines, symbol)
    return {'agent': 'Sentiment Researcher (Agent 2)', 'symbol': symbol,
            'market_type': market_type, 'headlines': headlines, **sentiment}


print('✅ Agent 2 (Sentiment Researcher) ready!')

---
## 🎯 Step 6: Agent 3 — Pattern & Strategy Engine
Deterministic playbook: Momentum Breakout / Contrarian Reversion / Capital Preservation.

In [ ]:
# ══════════════════════════════════════════════════════════
#  AGENT 3: PATTERN & STRATEGY ENGINE
# ══════════════════════════════════════════════════════════


def select_strategy(tech_data: dict, sent_data: dict) -> dict:
    """Select optimal strategy from playbook."""
    tech_score = tech_data.get('technical_score', 0)
    rsi = tech_data.get('indicators', {}).get('rsi_14')
    trend = tech_data.get('trend', 'SIDEWAYS')
    sent_score = sent_data.get('sentiment_score', 50)

    # Strategy 1: Momentum Breakout
    if trend == 'BULLISH' and rsi is not None and rsi < MOMENTUM_RSI_CEILING and sent_score > SENTIMENT_BULLISH_THRESHOLD:
        return {'strategy': 'MOMENTUM_BREAKOUT', 'action': 'BUY',
                'confidence': min(0.95, (tech_score + 1)/2 * 0.6 + sent_score/100 * 0.4),
                'reasoning': f'Bullish trend. RSI={rsi:.1f}. Sentiment={sent_score}/100. Score={tech_score:+.2f}. Momentum breakout.'}

    # Strategy 2: Contrarian Mean Reversion
    if rsi is not None and rsi < REVERSION_RSI_FLOOR and sent_score < SENTIMENT_BEARISH_THRESHOLD:
        return {'strategy': 'MEAN_REVERSION', 'action': 'BUY',
                'confidence': min(0.85, 0.5 + (30 - rsi)/100 + (40 - sent_score)/200),
                'reasoning': f'RSI={rsi:.1f} oversold. Sentiment={sent_score}/100 panic. Contrarian reversion.'}

    # Strategy 3: Bearish Trend Short
    if trend == 'BEARISH' and rsi is not None and rsi > REVERSION_RSI_FLOOR and sent_score < SENTIMENT_BEARISH_THRESHOLD and tech_score < -0.3:
        return {'strategy': 'TREND_FOLLOW_SHORT', 'action': 'SELL',
                'confidence': min(0.90, abs(tech_score) * 0.6 + (100 - sent_score)/100 * 0.4),
                'reasoning': f'Bearish trend. RSI={rsi:.1f}. Sentiment={sent_score}/100. Score={tech_score:+.2f}. Short setup.'}

    # Strategy 4: Capital Preservation
    return {'strategy': 'CAPITAL_PRESERVATION', 'action': 'HOLD', 'confidence': 0.0,
            'reasoning': f'No confluence. Trend={trend}. RSI={rsi}. Sentiment={sent_score}/100. Score={tech_score:+.2f}. Holding.'}


def compute_trade_levels(strat: dict, price: float, atr=None) -> dict:
    """Compute Entry, SL, TP levels."""
    action = strat.get('action', 'HOLD')
    if action == 'HOLD':
        return {'entry_price': None, 'stop_loss': None, 'take_profit': None, 'risk_reward_ratio': None}

    sl_dist = atr * 1.5 if atr and atr > 0 else price * DEFAULT_SL_PERCENT
    tp_dist = atr * 3.0 if atr and atr > 0 else price * DEFAULT_TP_PERCENT

    if action == 'BUY':
        sl, tp = round(price - sl_dist, 4), round(price + tp_dist, 4)
    else:
        sl, tp = round(price + sl_dist, 4), round(price - tp_dist, 4)

    risk = abs(price - sl)
    reward = abs(tp - price)
    rr = round(reward / risk, 2) if risk > 0 else 0
    return {'entry_price': round(price, 4), 'stop_loss': sl, 'take_profit': tp, 'risk_reward_ratio': rr}


def agent3_analyze(tech_data: dict, sent_data: dict, price: float) -> dict:
    """Full Agent 3 pipeline."""
    atr = tech_data.get('indicators', {}).get('atr_14')
    strat = select_strategy(tech_data, sent_data)
    levels = compute_trade_levels(strat, price, atr)
    return {'agent': 'Strategy Engine (Agent 3)', 'symbol': tech_data.get('symbol', ''),
            'current_price': round(price, 4), **strat, **levels}


print('✅ Agent 3 (Strategy Engine) ready!')

---
## 🛡️ Step 7: Agent 4 — Risk Guardian & Order Executor
The uncompromising 2% risk gatekeeper.

In [ ]:
# ══════════════════════════════════════════════════════════
#  AGENT 4: RISK GUARDIAN & ORDER EXECUTOR
# ══════════════════════════════════════════════════════════

from datetime import timezone

# Paper trading ledger
paper_portfolio = {
    'capital_inr': DEFAULT_CAPITAL_INR,
    'capital_usd': DEFAULT_CAPITAL_USD,
    'open_positions': [],
    'trade_history': [],
    'daily_pnl': 0.0,
}


def calculate_position_size(capital, entry, sl) -> dict:
    """Position Size = (Capital × 2%) / |Entry − SL|"""
    risk_per_unit = abs(entry - sl)
    if risk_per_unit <= 0:
        return {'position_size': 0, 'risk_amount': 0, 'risk_percent': 0, 'error': 'SL = Entry'}
    max_risk = capital * MAX_RISK_PER_TRADE
    pos_size = int(max_risk / risk_per_unit)
    actual_risk = pos_size * risk_per_unit
    return {'position_size': pos_size, 'risk_amount': round(actual_risk, 2),
            'risk_percent': round(actual_risk / capital * 100, 2) if capital > 0 else 0}


def validate_trade(proposal: dict, market_type: str = 'indian') -> dict:
    """Validate against all risk rules."""
    action = proposal.get('action', 'HOLD')
    entry, sl, tp = proposal.get('entry_price'), proposal.get('stop_loss'), proposal.get('take_profit')
    rr = proposal.get('risk_reward_ratio')
    vetoes = []

    if action == 'HOLD':
        return {'approved': False, 'action': 'HOLD', 'veto_reasons': ['HOLD — no trade.'], 'position_size': 0}
    if sl is None or entry is None:
        vetoes.append('VETO: Missing stop-loss or entry.')
    if rr is not None and rr < MIN_RISK_REWARD:
        vetoes.append(f'VETO: R:R {rr}:1 < minimum {MIN_RISK_REWARD}:1.')

    capital = paper_portfolio['capital_inr'] if market_type == 'indian' else paper_portfolio['capital_usd']
    if capital > 0 and abs(paper_portfolio['daily_pnl']) / capital > MAX_DAILY_DRAWDOWN:
        vetoes.append(f'VETO: Daily drawdown limit reached ({MAX_DAILY_DRAWDOWN*100}%).')

    if vetoes:
        return {'approved': False, 'action': action, 'veto_reasons': vetoes, 'position_size': 0}

    sizing = calculate_position_size(capital, entry, sl)
    if sizing.get('error') or sizing['position_size'] <= 0:
        return {'approved': False, 'action': action,
                'veto_reasons': [f"VETO: {sizing.get('error', 'Position size = 0')}"], 'position_size': 0}

    return {'approved': True, 'action': action, 'veto_reasons': [], **sizing,
            'entry_price': entry, 'stop_loss': sl, 'take_profit': tp, 'risk_reward_ratio': rr}


def execute_paper_trade(validated: dict, symbol: str, market_type: str = 'indian') -> dict:
    """Execute in Paper Trading mode."""
    if not validated.get('approved'):
        return {'executed': False, 'reason': 'Not approved.', 'veto_reasons': validated.get('veto_reasons', [])}

    ist = timedelta(hours=5, minutes=30)
    ts = datetime.now(timezone.utc) + ist
    trade = {
        'trade_id': f"PAPER-{ts.strftime('%Y%m%d%H%M%S')}-{symbol.replace('/', '-')}",
        'timestamp': ts.isoformat(), 'symbol': symbol, 'market_type': market_type,
        'action': validated['action'], 'entry_price': validated['entry_price'],
        'stop_loss': validated['stop_loss'], 'take_profit': validated['take_profit'],
        'position_size': validated['position_size'], 'risk_amount': validated['risk_amount'],
        'risk_percent': validated['risk_percent'], 'risk_reward_ratio': validated['risk_reward_ratio'],
        'status': 'OPEN', 'mode': 'PAPER_TRADING',
    }
    paper_portfolio['open_positions'].append(trade)
    paper_portfolio['trade_history'].append(trade)
    return {'executed': True, 'mode': 'PAPER_TRADING', 'trade_record': trade}


def agent4_analyze(proposal: dict, symbol: str, market_type: str = 'indian') -> dict:
    """Full Agent 4 pipeline."""
    validation = validate_trade(proposal, market_type)
    execution = {'executed': False, 'reason': 'Not approved.'}
    if validation['approved']:
        execution = execute_paper_trade(validation, symbol, market_type)

    capital_key = 'capital_inr' if market_type == 'indian' else 'capital_usd'
    currency = 'INR' if market_type == 'indian' else 'USD'
    portfolio = {'mode': EXECUTION_MODE, 'available_capital': paper_portfolio[capital_key],
                 'currency': currency, 'open_positions': len(paper_portfolio['open_positions']),
                 'total_trades': len(paper_portfolio['trade_history']), 'daily_pnl': paper_portfolio['daily_pnl']}

    return {'agent': 'Risk Guardian (Agent 4)', 'symbol': symbol, 'market_type': market_type,
            'risk_validation': validation, 'execution': execution, 'portfolio_status': portfolio}


print('✅ Agent 4 (Risk Guardian & Executor) ready!')

---
## 🚀 Step 8: Run the Full 4-Agent Pipeline!
Choose an Indian equity or Crypto pair and watch all 4 agents work together.

In [ ]:
# ══════════════════════════════════════════════════════════
#  FULL 4-AGENT PIPELINE
# ══════════════════════════════════════════════════════════

import json

def run_pipeline(symbol: str, market_type: str = 'indian'):
    """Execute the complete 4-agent trading pipeline."""
    print(f"\n{'='*60}")
    print(f"  🤖 MULTI-AGENT PIPELINE: {symbol} ({market_type.upper()})")
    print(f"{'='*60}")

    # Fetch data
    print(f"\n📊 Fetching OHLCV data...")
    if market_type == 'indian':
        df = fetch_indian_ohlcv(symbol, period_days=60)
        price = get_indian_price(symbol)
    else:
        df = fetch_crypto_ohlcv(symbol, timeframe='1d', limit=60)
        price = get_crypto_price(symbol)
    print(f"   ✅ {len(df)} candles. Price: {price}")

    # Agent 1
    print(f"\n🔬 Agent 1: Technical Analysis...")
    tech = agent1_analyze(df, symbol)
    print(f"   Trend: {tech['trend']} | RSI: {tech['indicators']['rsi_14']} | Score: {tech['technical_score']:+.2f}")
    print(f"   Patterns: {tech['candlestick_patterns'] or 'None'}")

    # Agent 2
    print(f"\n📰 Agent 2: Sentiment Analysis...")
    sent = agent2_analyze(symbol=symbol, market_type=market_type)
    print(f"   Score: {sent['sentiment_score']}/100 | Label: {sent['sentiment_label']}")
    print(f"   Headlines: {sent['headlines_analyzed']}")
    print(f"   Summary: {sent['narrative_summary']}")

    # Agent 3
    print(f"\n🎯 Agent 3: Strategy Selection...")
    strat = agent3_analyze(tech, sent, price)
    print(f"   Strategy: {strat['strategy']} | Action: {strat['action']} | Confidence: {strat.get('confidence', 0):.1%}")
    if strat.get('entry_price'):
        print(f"   Entry: {strat['entry_price']} | SL: {strat['stop_loss']} | TP: {strat['take_profit']} | R:R: 1:{strat['risk_reward_ratio']}")
    print(f"   Reasoning: {strat['reasoning']}")

    # Agent 4
    print(f"\n🛡️ Agent 4: Risk Validation & Execution...")
    risk = agent4_analyze(strat, symbol, market_type)
    v = risk['risk_validation']
    if v['approved']:
        print(f"   ✅ APPROVED | Size: {v['position_size']} units | Risk: {v['risk_amount']} ({v['risk_percent']}%)")
        ex = risk['execution']
        if ex.get('executed'):
            print(f"   ✅ EXECUTED (Paper) | Trade ID: {ex['trade_record']['trade_id']}")
    else:
        print(f"   ❌ VETOED:")
        for r in v.get('veto_reasons', []): print(f"      • {r}")

    p = risk['portfolio_status']
    print(f"\n💰 Portfolio: {p['available_capital']} {p['currency']} | Open: {p['open_positions']} | Trades: {p['total_trades']}")
    print(f"{'='*60}\n")

    return {'tech': tech, 'sentiment': sent, 'strategy': strat, 'risk': risk}

### 📈 Test: Indian Equity (RELIANCE.NS)

In [ ]:
# Run pipeline on RELIANCE.NS (Indian Market)
result_reliance = run_pipeline("RELIANCE.NS", "indian")

### ₿ Test: Crypto (BTC/USDT)

In [ ]:
# Run pipeline on BTC/USDT (Crypto Market)
result_btc = run_pipeline("BTC/USDT", "crypto")

### 🔄 Full Watchlist Scan

In [ ]:
# Scan all Indian equities in the watchlist
print("📈 SCANNING INDIAN EQUITIES WATCHLIST")
print("=" * 60)
for symbol in INDIAN_WATCHLIST:
    try:
        run_pipeline(symbol, 'indian')
    except Exception as e:
        print(f"⚠️ Error: {symbol} — {e}")

In [ ]:
# Scan all crypto pairs in the watchlist
print("₿ SCANNING CRYPTO WATCHLIST")
print("=" * 60)
for symbol in CRYPTO_WATCHLIST:
    try:
        run_pipeline(symbol, 'crypto')
    except Exception as e:
        print(f"⚠️ Error: {symbol} — {e}")

### 💰 Portfolio Summary

In [ ]:
# Print paper trading portfolio summary
print("\n💰 PAPER TRADING PORTFOLIO SUMMARY")
print("=" * 60)
print(f"Indian Capital: ₹{paper_portfolio['capital_inr']:,.2f}")
print(f"Crypto Capital: ${paper_portfolio['capital_usd']:,.2f}")
print(f"Open Positions: {len(paper_portfolio['open_positions'])}")
print(f"Total Trades: {len(paper_portfolio['trade_history'])}")
print(f"Daily P&L: {paper_portfolio['daily_pnl']}")

if paper_portfolio['trade_history']:
    print("\n📋 Trade History:")
    for t in paper_portfolio['trade_history']:
        print(f"  {t['trade_id']} | {t['action']} {t['symbol']} | Entry: {t['entry_price']} | SL: {t['stop_loss']} | TP: {t['take_profit']} | Size: {t['position_size']}")